# Annotation agreement

Evaluate agreement between Victoria and Gabriel using Cohen's kappa. Relevance is evaluated over all topics; name-option agreement is evaluated only where both annotators marked the topic as relevant. A review sheet is generated from the union of relevance and name disagreements.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import cohen_kappa_score

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
ANNOTATORS_DIR = PROJECT_DIR / 'results/manual_annotation/annotators'
VICTORIA_PATH = ANNOTATORS_DIR / 'topic_annotation_victoria.csv'
GABRIEL_PATH = ANNOTATORS_DIR / 'topic_annotation_Gabriel.csv'
OUTPUT_DIR = PROJECT_DIR / 'results/annotation_agreement'
REVIEW_PATH = OUTPUT_DIR / 'third_annotator_review.csv'
SUMMARY_PATH = OUTPUT_DIR / 'agreement_summary.csv'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ANNOTATION_FIELDS = ['relevance', 'selected_name_option', 'manual_name', 'notes', 'annotator_id', 'annotated_at']
ALLOWED_RELEVANCE = {'relevante', 'irrelevante'}
ALLOWED_NAME_OPTIONS = {'1', '2', '3', 'manual'}

In [ ]:
def load_annotation(path, expected_annotator):
    if not path.exists():
        raise FileNotFoundError(path)
    frame = pd.read_csv(path, dtype=str).fillna('')
    if frame['topic_id'].duplicated().any():
        raise ValueError(f'{path.name}: topic_id duplicado.')
    if not set(frame['relevance']).issubset(ALLOWED_RELEVANCE) or frame['relevance'].eq('').any():
        raise ValueError(f'{path.name}: relevância ausente ou inválida.')
    relevant = frame['relevance'].eq('relevante')
    if not set(frame.loc[relevant, 'selected_name_option']).issubset(ALLOWED_NAME_OPTIONS):
        raise ValueError(f'{path.name}: opção de nome inválida.')
    manual_without_name = relevant & frame['selected_name_option'].eq('manual') & frame['manual_name'].str.strip().eq('')
    if manual_without_name.any():
        raise ValueError(f'{path.name}: nome manual ausente.')
    if not frame['annotator_id'].eq(expected_annotator).all():
        raise ValueError(f'{path.name}: annotator_id inesperado.')
    return frame.sort_values('topic_id', key=lambda values: values.astype(int)).reset_index(drop=True)

victoria = load_annotation(VICTORIA_PATH, 'victoria')
gabriel = load_annotation(GABRIEL_PATH, 'Gabriel')
if not victoria['topic_id'].equals(gabriel['topic_id']):
    raise ValueError('Os anotadores não avaliaram os mesmos topic_id.')
static_columns = [column for column in victoria.columns if column not in ANNOTATION_FIELDS]
if not victoria[static_columns].equals(gabriel[static_columns]):
    raise ValueError('Exemplos, sugestões ou metadados diferem entre os anotadores.')
print(f'{len(victoria)} tópicos completos e alinhados')

In [ ]:
def selected_name(row):
    if row['relevance'] != 'relevante':
        return ''
    option = row['selected_name_option']
    return row['manual_name'].strip() if option == 'manual' else row[f'suggestion_{option}'].strip()

victoria['selected_name'] = victoria.apply(selected_name, axis=1)
gabriel['selected_name'] = gabriel.apply(selected_name, axis=1)

relevance_agreement = victoria['relevance'].eq(gabriel['relevance'])
relevance_kappa = cohen_kappa_score(victoria['relevance'], gabriel['relevance'])
relevance_table = pd.crosstab(victoria['relevance'], gabriel['relevance'], rownames=['Victoria'], colnames=['Gabriel'], dropna=False)

both_relevant = victoria['relevance'].eq('relevante') & gabriel['relevance'].eq('relevante')
name_option_agreement = victoria.loc[both_relevant, 'selected_name_option'].eq(gabriel.loc[both_relevant, 'selected_name_option'])
name_option_kappa = cohen_kappa_score(victoria.loc[both_relevant, 'selected_name_option'], gabriel.loc[both_relevant, 'selected_name_option'])
name_table = pd.crosstab(victoria.loc[both_relevant, 'selected_name_option'], gabriel.loc[both_relevant, 'selected_name_option'], rownames=['Victoria'], colnames=['Gabriel'], dropna=False)

relevance_disagreement = ~relevance_agreement
name_disagreement = both_relevant & ~victoria['selected_name'].str.casefold().eq(gabriel['selected_name'].str.casefold())
needs_review = relevance_disagreement | name_disagreement

summary = pd.DataFrame([
    {'dimension': 'relevance', 'n_evaluated': len(victoria), 'n_agreements': int(relevance_agreement.sum()), 'n_disagreements': int(relevance_disagreement.sum()), 'observed_agreement': relevance_agreement.mean(), 'cohen_kappa': relevance_kappa},
    {'dimension': 'name_option_both_relevant', 'n_evaluated': int(both_relevant.sum()), 'n_agreements': int(name_option_agreement.sum()), 'n_disagreements': int((~name_option_agreement).sum()), 'observed_agreement': name_option_agreement.mean(), 'cohen_kappa': name_option_kappa},
])
summary.to_csv(SUMMARY_PATH, index=False)
relevance_table.to_csv(OUTPUT_DIR / 'relevance_confusion_matrix.csv')
name_table.to_csv(OUTPUT_DIR / 'name_option_confusion_matrix.csv')
display(summary)
display(relevance_table)
display(name_table)
print(f'Tópicos para revisão: {needs_review.sum()}')

In [ ]:
review = victoria.loc[needs_review, static_columns].copy()
review.insert(1, 'disagreement_type', np.select(
    [relevance_disagreement[needs_review], name_disagreement[needs_review]],
    ['relevance', 'name'],
    default='name',
))
review['victoria_relevance'] = victoria.loc[needs_review, 'relevance'].to_numpy()
review['victoria_name_option'] = victoria.loc[needs_review, 'selected_name_option'].to_numpy()
review['victoria_selected_name'] = victoria.loc[needs_review, 'selected_name'].to_numpy()
review['victoria_notes'] = victoria.loc[needs_review, 'notes'].to_numpy()
review['gabriel_relevance'] = gabriel.loc[needs_review, 'relevance'].to_numpy()
review['gabriel_name_option'] = gabriel.loc[needs_review, 'selected_name_option'].to_numpy()
review['gabriel_selected_name'] = gabriel.loc[needs_review, 'selected_name'].to_numpy()
review['gabriel_notes'] = gabriel.loc[needs_review, 'notes'].to_numpy()
for field in ANNOTATION_FIELDS:
    review[field] = ''

if REVIEW_PATH.exists():
    previous = pd.read_csv(REVIEW_PATH, dtype=str).fillna('').set_index('topic_id')
    for field in ANNOTATION_FIELDS:
        review[field] = review['topic_id'].map(previous[field]).fillna('')

review = review.sort_values(['disagreement_type', 'topic_id'], key=lambda column: column.astype(int) if column.name == 'topic_id' else column).reset_index(drop=True)
review.to_csv(REVIEW_PATH, index=False)
print(REVIEW_PATH)
display(review[['topic_id', 'disagreement_type', 'victoria_relevance', 'gabriel_relevance', 'victoria_selected_name', 'gabriel_selected_name']].head(10))

The review CSV is compatible with `scripts/annotate_topics.py`. Example:

```bash
/scratch/victoria.estanislau/g2/bin/python scripts/annotate_topics.py \
  --annotator-id terceiro_anotador \
  --base-csv results/annotation_agreement/third_annotator_review.csv \
  --output-dir results/annotation_agreement/annotators
```

The terminal interface presents the same ten questions and three names. Earlier decisions remain in the CSV for later adjudication but are not displayed by the terminal script.